# VINGAGE – Notebook 3: robustness to darkened input, with and without CLAHE (objective O4a)
Takes a fixed subset of **Test** clips (all level 0–1 clips plus random others, 600 in total), darkens every frame to 25% brightness, and extracts features twice: **without CLAHE** and **with CLAHE**. Models are trained on the clean Train features from notebook 1 (`features_all.csv`) and scored on clean vs darkened versions of the same clips.

Runtime: CPU. Needs the DAiSEE videos on the Colab disk again (cell 4, about 10 min). Expect roughly 1–1.5 hours for the two extraction runs. Resumable: if Colab disconnects, reconnect and re-run all cells; finished chunks are skipped.

## 1. Install

In [ ]:
!pip -q install mediapipe xgboost shap
!wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!ls -la /content/face_landmarker.task

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/daisee_project'
os.makedirs(WORK, exist_ok=True)
print('outputs go to', WORK)

## 3. Project code (updated modules with darken / CLAHE switches)

In [ ]:
%%writefile /content/ed_features.py
"""Per-clip feature extraction for DAiSEE. No pixels are written to disk."""
import numpy as np, cv2

R_EYE = [33, 160, 158, 133, 153, 144]     # p1..p6, right eye (image left)
L_EYE = [362, 385, 387, 263, 373, 380]    # p1..p6, left eye
R_IRIS, L_IRIS = 468, 473
BLEND = ["eyeBlinkLeft", "eyeBlinkRight", "eyeLookInLeft", "eyeLookOutLeft", "eyeLookInRight",
         "eyeLookOutRight", "eyeLookUpLeft", "eyeLookDownLeft", "browDownLeft", "browDownRight",
         "browInnerUp", "jawOpen", "mouthSmileLeft", "mouthSmileRight"]
SIGNALS = ["yaw", "pitch", "roll", "ear", "gaze_x", "gaze_y", "motion"] + BLEND

def eye_aspect_ratio(eye):
    v1 = np.linalg.norm(eye[1] - eye[5]); v2 = np.linalg.norm(eye[2] - eye[4])
    h = np.linalg.norm(eye[0] - eye[3])
    return (v1 + v2) / (2.0 * h + 1e-8)

def head_pose_deg(M):
    R = M[:3, :3] / np.linalg.norm(M[:3, :3], axis=0)
    sy = np.hypot(R[0, 0], R[1, 0])
    pitch = np.degrees(np.arctan2(R[2, 1], R[2, 2]))
    yaw = np.degrees(np.arctan2(-R[2, 0], sy))
    roll = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    return yaw, pitch, roll

def clahe_if_dark(bgr, thresh=90, clip=2.0):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    if lab[..., 0].mean() >= thresh:
        return bgr, False
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), True

def frame_signals(res, w, h):
    """res: FaceLandmarkerResult. Returns dict of signals or None if no face."""
    if not res.face_landmarks:
        return None
    lm = res.face_landmarks[0]
    pts = np.array([[p.x * w, p.y * h] for p in lm])
    out = {}
    out["yaw"], out["pitch"], out["roll"] = head_pose_deg(np.asarray(res.facial_transformation_matrixes[0]))
    out["ear"] = 0.5 * (eye_aspect_ratio(pts[R_EYE]) + eye_aspect_ratio(pts[L_EYE]))
    gx, gy = [], []
    for iris, (a, b, up, lo) in ((R_IRIS, (33, 133, 159, 145)), (L_IRIS, (362, 263, 386, 374))):
        c = 0.5 * (pts[a] + pts[b]); wid = np.linalg.norm(pts[a] - pts[b]) + 1e-8
        gx.append((pts[iris][0] - c[0]) / wid)
        gy.append((pts[iris][1] - 0.5 * (pts[up][1] + pts[lo][1])) / wid)
    out["gaze_x"], out["gaze_y"] = float(np.mean(gx)), float(np.mean(gy))
    bs = {c.category_name: c.score for c in res.face_blendshapes[0]} if res.face_blendshapes else {}
    for k in BLEND:
        out[k] = float(bs.get(k, np.nan))
    x0, y0 = np.clip(pts.min(0), 0, [w - 1, h - 1]); x1, y1 = np.clip(pts.max(0), 0, [w - 1, h - 1])
    out["_bbox"] = (int(x0), int(y0), int(x1), int(y1))
    return out

def face_gray(bgr, bbox, size=64):
    x0, y0, x1, y1 = bbox
    if x1 - x0 < 8 or y1 - y0 < 8:
        return None
    return cv2.resize(cv2.cvtColor(bgr[y0:y1, x0:x1], cv2.COLOR_BGR2GRAY), (size, size))

def median3(a):
    a = np.asarray(a, float)
    if len(a) < 3:
        return a
    out = a.copy()
    for i in range(1, len(a) - 1):
        win = a[i - 1:i + 2]; win = win[~np.isnan(win)]
        out[i] = np.median(win) if len(win) else np.nan
    return out

def extract_clip(path, landmarker, target_fps=5, darken=None, use_clahe=True):
    """Return (T, len(SIGNALS)) array with NaN rows where no face; plus n_frames and n_clahe."""
    import mediapipe as mp
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(1, int(round(fps / target_fps)))
    rows, prev, i, n_clahe = [], None, 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % step == 0:
            if darken:                                    # simulate a dark room: scale pixel values
                frame = np.clip(frame.astype(np.float32) * darken, 0, 255).astype(np.uint8)
            if use_clahe:
                frame, used = clahe_if_dark(frame); n_clahe += used
            h, w = frame.shape[:2]
            res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            sig = frame_signals(res, w, h)
            if sig is None:
                rows.append([np.nan] * len(SIGNALS)); prev = None
            else:
                g = face_gray(frame, sig["_bbox"])
                motion = np.nan
                if g is not None and prev is not None:
                    flow = cv2.calcOpticalFlowFarneback(prev, g, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                    motion = float(np.linalg.norm(flow, axis=2).mean())
                prev = g
                sig["motion"] = motion
                rows.append([sig[k] for k in SIGNALS])
        i += 1
    cap.release()
    A = np.array(rows, float).reshape(-1, len(SIGNALS))
    det = ~np.isnan(A[:, 0]) if len(A) else np.zeros(0, bool)   # raw detections, before filtering
    for j in range(A.shape[1]):
        A[:, j] = median3(A[:, j])
    return A, det, n_clahe

def window_features(A, det, ear_closed=0.2):
    """One row of features for one 10 s clip (the whole clip is one window). det = raw detection mask."""
    T = len(A)
    feats = {"n_frames": T, "detect_rate": float(det.mean()) if T else 0.0}
    t = np.arange(T)
    for j, name in enumerate(SIGNALS):
        x = A[:, j] if T else np.array([])
        m = ~np.isnan(x)
        if m.sum() >= 3:
            feats[f"{name}_mean"] = x[m].mean(); feats[f"{name}_std"] = x[m].std()
            feats[f"{name}_min"] = x[m].min(); feats[f"{name}_max"] = x[m].max()
            feats[f"{name}_slope"] = np.polyfit(t[m], x[m], 1)[0]
        else:
            for s in ("mean", "std", "min", "max", "slope"):
                feats[f"{name}_{s}"] = np.nan
    ear = A[:, SIGNALS.index("ear")] if T else np.array([])
    feats["perclos"] = float((ear[~np.isnan(ear)] < ear_closed).mean()) if (~np.isnan(ear)).any() else np.nan
    return feats

def aggregate(probs, k_min=3):
    if len(probs) < k_min:
        return None
    p = np.asarray(probs)
    return {"mean": float(p.mean()), "share_engaged": float((p >= 0.5).mean()), "n": len(p)}


In [ ]:
%%writefile /content/ed_pipeline.py
"""Index, audit and resumable extraction for DAiSEE."""
import os, re, glob, time
from pathlib import Path
import numpy as np, pandas as pd

VIDEO_EXT = {".avi", ".mp4", ".mov", ".mkv"}
_FACTORY = None; _LM = None; _SEQDIR = None; _OPTS = {}

def build_index(root):
    """Return DataFrame: clip, path, split, subject, Boredom/Engagement/Confusion/Frustration."""
    vids = {}
    for dp, _, fs in os.walk(root):
        for f in fs:
            if Path(f).suffix.lower() in VIDEO_EXT:
                vids[Path(f).stem] = os.path.join(dp, f)
    labs = []
    for p in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        n = os.path.basename(p).lower()
        if "label" not in n or "all" in n:
            continue
        split = "Train" if "train" in n else "Validation" if ("valid" in n or "val" in n) else "Test" if "test" in n else None
        if split is None:
            continue
        d = pd.read_csv(p); d.columns = [c.strip() for c in d.columns]; d["split"] = split; labs.append(d)
    if not labs:
        raise RuntimeError("No Train/Validation/Test label CSVs found under " + root)
    L = pd.concat(labs, ignore_index=True)
    L["clip"] = L["ClipID"].astype(str).map(lambda x: Path(x.strip()).stem)
    L["path"] = L["clip"].map(vids)
    L["subject"] = L["clip"].str[:6]
    return L.drop(columns=["ClipID"]), len(vids)

def audit(L, n_videos):
    print(f"video files found: {n_videos}; labelled rows: {len(L)}; rows with a video: {L['path'].notna().sum()}")
    print("\nclips per split:\n", L["split"].value_counts())
    print("\nEngagement level share per split (%):")
    print((pd.crosstab(L["split"], L["Engagement"], normalize="index") * 100).round(2))
    subs = {s: set(L.loc[L.split == s, "subject"]) for s in L.split.unique()}
    names = list(subs)
    print("\nsubjects per split:", {k: len(v) for k, v in subs.items()})
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            print(f"subjects shared by {names[i]} and {names[j]}:", len(subs[names[i]] & subs[names[j]]))
    print("\n(subject = first 6 characters of the clip ID: an assumption, check that the counts above look like ~112 users)")

def _init(factory, seqdir, opts):
    global _LM, _SEQDIR, _OPTS
    _LM = factory(); _SEQDIR = seqdir; _OPTS = opts

def _work(args):
    import ed_features as E
    clip, path = args
    t = time.time()
    try:
        A, det, nc = E.extract_clip(path, _LM, darken=_OPTS.get("darken"), use_clahe=_OPTS.get("use_clahe", True))
        seq = np.concatenate([A, det[:, None].astype(float)], axis=1).astype(np.float32)   # signals + raw detection flag
        np.save(os.path.join(_SEQDIR, clip + ".npy"), seq)
        f = E.window_features(A, det); f["clahe_frames"] = nc; f["error"] = ""
    except Exception as e:
        f = {"error": repr(e)[:200]}
    f["clip"] = clip; f["secs"] = time.time() - t
    return f

def run_extraction(L, out_dir, factory, workers=2, chunk=100, limit=None, darken=None, use_clahe=True):
    """Resumable: each chunk of clips is saved to out_dir/chunk_XXXX.csv and skipped if already present."""
    import multiprocessing as mp
    os.makedirs(out_dir, exist_ok=True)
    seqdir = os.path.join(out_dir, "seq"); os.makedirs(seqdir, exist_ok=True)
    todo = L.dropna(subset=["path"]).reset_index(drop=True)
    if limit: todo = todo.iloc[:limit]
    n_chunks = (len(todo) + chunk - 1) // chunk
    ctx = mp.get_context("fork")
    with ctx.Pool(workers, initializer=_init, initargs=(factory, seqdir, {"darken": darken, "use_clahe": use_clahe})) as pool:
        for c in range(n_chunks):
            fn = os.path.join(out_dir, f"chunk_{c:04d}.csv")
            if os.path.exists(fn):
                continue
            part = todo.iloc[c * chunk:(c + 1) * chunk]
            t = time.time()
            rows = pool.map(_work, list(zip(part["clip"], part["path"])), chunksize=2)
            pd.DataFrame(rows).to_csv(fn + ".tmp", index=False); os.replace(fn + ".tmp", fn)
            print(f"chunk {c + 1}/{n_chunks} done in {time.time() - t:.0f}s", flush=True)

def collect(L, out_dir):
    F = pd.concat([pd.read_csv(p, dtype={"clip": str}) for p in sorted(glob.glob(os.path.join(out_dir, "chunk_*.csv")))], ignore_index=True)
    F["error"] = F["error"].fillna("")
    return L.drop(columns=["path"]).merge(F, on="clip", how="inner")


In [ ]:
%%writefile /content/ed_models.py
"""Baselines on the clip feature table. Reports accuracy, macro-F1 and per-class recall."""
import numpy as np, pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight

NON_FEATURE = {"clip", "split", "subject", "Boredom", "Engagement", "Confusion", "Frustration", "error", "secs", "clahe_frames"}

def feature_cols(F):
    return [c for c in F.columns if c not in NON_FEATURE and pd.api.types.is_numeric_dtype(F[c])]

def make_xy(F, task):
    y = F["Engagement"].astype(int)
    if task == "binary":
        y = (y >= 2).astype(int)
    return y

def run_baselines(F, task="four", min_detect=0.0, seed=0):
    from xgboost import XGBClassifier
    F = F[(F["error"] == "") & (F["detect_rate"] >= min_detect)]
    cols = feature_cols(F)
    tr, va, te = (F[F.split == s] for s in ("Train", "Validation", "Test"))
    med = tr[cols].median()
    X = {k: d[cols].fillna(med).fillna(0).values for k, d in (("val", va), ("test", te), ("train", tr))}
    ytr = make_xy(tr, task).values; classes = sorted(set(ytr)); remap = {c: i for i, c in enumerate(classes)}
    labels = [0, 1] if task == "binary" else [0, 1, 2, 3]
    models = {
        "majority": DummyClassifier(strategy="most_frequent"),
        "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
        "xgboost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                 random_state=seed, n_jobs=2, eval_metric="mlogloss"),
    }
    out, fitted = [], {}
    for name, m in models.items():
        yt = np.array([remap[v] for v in ytr])
        if name == "xgboost":
            m.fit(X["train"], yt, sample_weight=compute_sample_weight("balanced", yt))
        else:
            m.fit(X["train"], yt)
        fitted[name] = m
        for split, d in (("Validation", va), ("Test", te)):
            y = make_xy(d, task).values
            p = np.array([classes[i] for i in m.predict(X["val" if split == "Validation" else "test"])])
            row = {"model": name, "split": split, "n": len(y), "accuracy": accuracy_score(y, p),
                   "macro_F1": f1_score(y, p, labels=labels, average="macro", zero_division=0)}
            for l, r in zip(labels, recall_score(y, p, labels=labels, average=None, zero_division=0)):
                row[f"recall_{l}"] = r
            out.append(row)
    return pd.DataFrame(out).round(3), fitted, cols


def fit_models(F, task="four", seed=0):
    """Train the three feature models on the Train split and return what is needed to score other feature tables."""
    from xgboost import XGBClassifier
    F = F[F["error"] == ""]
    cols = feature_cols(F); tr = F[F.split == "Train"]; med = tr[cols].median()
    ytr = make_xy(tr, task).values; classes = sorted(set(ytr)); remap = {c: i for i, c in enumerate(classes)}
    yt = np.array([remap[v] for v in ytr]); Xtr = tr[cols].fillna(med).fillna(0).values
    models = {
        "majority": DummyClassifier(strategy="most_frequent"),
        "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
        "xgboost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                 random_state=seed, n_jobs=2, eval_metric="mlogloss"),
    }
    for name, mod in models.items():
        if name == "xgboost": mod.fit(Xtr, yt, sample_weight=compute_sample_weight("balanced", yt))
        else: mod.fit(Xtr, yt)
    return {"models": models, "cols": cols, "med": med, "classes": classes, "task": task}

def score(bundle, Fe, condition):
    """Score a feature table with already-trained models. Returns accuracy and macro-F1 per model."""
    task, cols, med, classes = bundle["task"], bundle["cols"], bundle["med"], bundle["classes"]
    Fe = Fe[Fe["error"] == ""]; X = Fe[cols].fillna(med).fillna(0).values; y = make_xy(Fe, task).values
    labels = [0, 1] if task == "binary" else [0, 1, 2, 3]; rows = []
    for name, mod in bundle["models"].items():
        p = np.array([classes[i] for i in mod.predict(X)])
        rows.append({"task": task, "condition": condition, "model": name, "n": len(y), "face_detect_rate": round(Fe["detect_rate"].mean(), 3),
                     "accuracy": accuracy_score(y, p), "macro_F1": f1_score(y, p, labels=labels, average="macro", zero_division=0)})
    return pd.DataFrame(rows).round(3)


## 4. Get DAiSEE onto the Colab disk (same as notebook 1; set FILE_ID or ZIP_ON_DRIVE again)

In [ ]:
import os
ZIP_ON_DRIVE = '/content/drive/MyDrive/DAiSEE.zip'   # path of the shortcut or copy in your Drive
FILE_ID = ''                                           # alternative: file id from the share link

if os.path.exists('/content/daisee') and os.listdir('/content/daisee'):
    print('already extracted')
else:
    if os.path.exists(ZIP_ON_DRIVE):
        os.system(f'cp "{ZIP_ON_DRIVE}" /content/DAiSEE.zip')
    elif FILE_ID:
        os.system('pip -q install gdown'); os.system(f'gdown --id {FILE_ID} -O /content/DAiSEE.zip')
    else:
        raise SystemExit('Set ZIP_ON_DRIVE or FILE_ID first')
    os.system('unzip -q -o /content/DAiSEE.zip -d /content/daisee && rm /content/DAiSEE.zip')
os.system('df -h /content | tail -1')
for dp, dn, fn in os.walk('/content/daisee'):
    if dp.count(os.sep) <= 4: print(dp, '->', len(dn), 'folders,', len(fn), 'files')

## 5. Index and choose the 600-clip Test subset

In [ ]:
import sys; sys.path.insert(0, '/content')
import numpy as np, pandas as pd, ed_pipeline as P, ed_models as M
L, n_videos = P.build_index('/content/daisee')
T = L[(L.split == 'Test') & L.path.notna()]
low = T[T.Engagement.astype(int) <= 1]
rest = T.drop(low.index).sample(n=600 - len(low), random_state=0)
SUB = pd.concat([low, rest]).sort_values('clip').reset_index(drop=True)
print(len(SUB), 'clips;', 'level counts:', SUB.Engagement.astype(int).value_counts().sort_index().to_dict())
SUB[['clip']].to_csv(f'{WORK}/dark_subset_clips.csv', index=False)

## 6. Extraction: darkened (x0.25), CLAHE OFF, then darkened, CLAHE ON

In [ ]:
import shutil, time

def make_landmarker():
    import mediapipe as mp
    from mediapipe.tasks import python as mpp
    from mediapipe.tasks.python import vision
    opts = vision.FaceLandmarkerOptions(
        base_options=mpp.BaseOptions(model_asset_path='/content/face_landmarker.task'),
        running_mode=vision.RunningMode.IMAGE, num_faces=1,
        output_face_blendshapes=True, output_facial_transformation_matrixes=True)
    return vision.FaceLandmarker.create_from_options(opts)


P.run_extraction(SUB, f'{WORK}/dark_noclahe', make_landmarker, workers=2, chunk=100, darken=0.25, use_clahe=False)
P.run_extraction(SUB, f'{WORK}/dark_clahe', make_landmarker, workers=2, chunk=100, darken=0.25, use_clahe=True)
D0 = P.collect(SUB, f'{WORK}/dark_noclahe'); D1 = P.collect(SUB, f'{WORK}/dark_clahe')
print('errors:', (D0.error != '').sum(), (D1.error != '').sum())

## 7. Score: clean vs dark vs dark + CLAHE (models trained on clean Train only)

In [ ]:
F = pd.read_csv(f'{WORK}/features_all.csv', dtype={'clip': str}); F['error'] = F['error'].fillna('')
Fc = F[F.clip.isin(SUB.clip)]
out = []
for task in ('four', 'binary'):
    B = M.fit_models(F, task=task)
    for cond, Fe in (('clean', Fc), ('dark_noCLAHE', D0), ('dark_CLAHE', D1)):
        out.append(M.score(B, Fe, cond))
R = pd.concat(out, ignore_index=True)
R.to_csv(f'{WORK}/dark_results.csv', index=False)
print(R.to_string(index=False))

**Share the printed table from cell 7** (task / condition / model / n / face_detect_rate / accuracy / macro_F1) and the `errors:` line and subset counts from cells 5–6. Table 7 of the report is filled only from this.